# Hawkes/SVMHJD Continuous Comparator

This notebook is a guarded public workflow for the Hawkes/SVMHJD continuous TC-VAE comparator. The benchmark represents paths in signed log-return space, so the registered repaired comparator uses an identity transform and an identity inverse transform rather than a positive-price transform. The dataset is a scenario-data stress test for clustered marked jumps and lower-tail behaviour; it is not an arbitrage-free pricing model.

The notebook follows the original TC-VAE workflow shape: inspect the selected model, print reproducible commands, and optionally analyse local evaluation artefacts. It does not train or evaluate full models by default, does not require local checkpoints, and keeps all generated files under ignored `outputs/` paths.


## Setup

Locate the repository, load common helpers, and keep optional plotting imports guarded for lightweight execution.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "continuous"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display

try:
    from torchview import draw_graph
except ImportError:
    draw_graph = None

AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "../../trained_models/model_registry.yaml"
RUN_SMOKE = True
RUN_FULL = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False

EXPERIMENT_ID = "hawkes_jump"
FAMILY = "continuous"
N_SAMPLE_SMOKE = 128
N_SAMPLE_FULL = 1024
N_SAMPLE = N_SAMPLE_FULL if RUN_FULL else N_SAMPLE_SMOKE

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

print(f"repo_root={REPO_ROOT}")
print(f"torchview_available={draw_graph is not None}")
print(f"run_training={RUN_TRAINING}")
print(f"run_evaluation={RUN_EVALUATION}")


In [ ]:
def repo_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def resolve_parameter_path(path: str | Path) -> Path:
    raw_path = Path(path)
    if raw_path.is_absolute():
        return raw_path
    notebook_relative = (NOTEBOOK_DIR / raw_path).resolve()
    if notebook_relative.exists():
        return notebook_relative
    return (REPO_ROOT / raw_path).resolve()


def display_path(path: str | Path) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def shell_join(parts: Sequence[str | Path]) -> str:
    return shlex.join([str(part) for part in parts])


def read_yaml(path: str | Path) -> dict[str, Any]:
    with repo_path(path).open("r", encoding="utf-8") as handle:
        payload = yaml.safe_load(handle) or {}
    return payload if isinstance(payload, dict) else {}


def read_json_if_present(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    with resolved.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    return payload if isinstance(payload, dict) else None


def load_torch_mapping_if_present(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not (RUN_HEAVY and resolved.exists()):
        return None
    payload = torch.load(resolved, map_location="cpu", weights_only=True)
    return dict(payload) if isinstance(payload, Mapping) else None


def metric_value(metrics: Mapping[str, Any], *keys: str) -> float | None:
    for key in keys:
        value = metrics.get(key)
        if isinstance(value, Mapping):
            value = value.get("mean")
        if isinstance(value, int | float):
            return float(value)
    return None


def metric_std(metrics: Mapping[str, Any], *keys: str) -> float | None:
    for key in keys:
        value = metrics.get(key)
        if isinstance(value, Mapping) and isinstance(value.get("std"), int | float):
            return float(value["std"])
        std_value = metrics.get(f"{key}_std")
        if isinstance(std_value, int | float):
            return float(std_value)
    return None


def format_metric(metrics: Mapping[str, Any], *keys: str) -> str:
    value = metric_value(metrics, *keys)
    std = metric_std(metrics, *keys)
    if value is None:
        return ""
    return f"{value:.4g}" if std is None else f"{value:.4g} +/- {std:.2g}"


def tensor_2d(values: torch.Tensor) -> torch.Tensor:
    tensor = values.detach().cpu().float()
    if tensor.ndim == 3 and tensor.shape[-1] == 1:
        tensor = tensor[..., 0]
    if tensor.ndim != 2:
        raise ValueError(f"Expected [batch, time] or [batch, time, 1], got {tuple(values.shape)}")
    return tensor


## Registry Selection

Select the continuous Hawkes/SVMHJD candidate from `trained_models/model_registry.yaml` and display its config and local checkpoint convention. Local checkpoint paths are conventions only; they are not required for this notebook to run.


In [ ]:
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

registry_path = resolve_parameter_path(MODEL_REGISTRY_PATH)
registry = load_registry(registry_path) if AUTO_SELECT_MODEL else {}
selected = select_registered_model(registry, experiment=EXPERIMENT_ID, family=FAMILY)

CONFIG_PATH = Path(selected.config or "configs/experiments/hawkes_jump_beta_cvae_logreturn_identity.yaml")
CHECKPOINT_CONVENTION = selected.checkpoint_paths.get("checkpoint_convention", "local_outputs_only")
CONTINUOUS_MODEL_DIR = Path("outputs/hawkes_jump_continuous_logreturn_identity/seed0/final_model")
EVALUATION_DIR = Path("outputs/hawkes_jump_continuous_logreturn_identity/seed0/evaluation")

selection_row = {
    "experiment": selected.experiment,
    "family": selected.family,
    "candidate": selected.candidate_id,
    "selected_by": selected.selected_by,
    "selection_profile": selected.selection_profile,
    "config": selected.config,
    "checkpoint_convention": CHECKPOINT_CONVENTION,
    "status": selected.candidate.get("status"),
    "public_default": selected.candidate.get("public_default"),
}
display(pd.DataFrame([selection_row]))

if selected.missing_metrics:
    display(Markdown("Missing metrics: " + ", ".join(f"`{item}`" for item in selected.missing_metrics)))


## Config Inspection

The registered comparator uses `hawkes_jump_beta_cvae_logreturn_identity.yaml`. The accompanying InfoCVAE identity configs are present for ablation, but the registry-selected continuous comparator remains the repaired BetaCVAE entry.


In [ ]:
config_payload = read_yaml(CONFIG_PATH)
rows: list[dict[str, Any]] = []
for section_name in ("experiment", "data", "model", "training"):
    section = config_payload.get(section_name, {})
    if isinstance(section, Mapping):
        for key, value in section.items():
            if isinstance(value, Mapping):
                continue
            rows.append({"section": section_name, "key": key, "value": value})

display(pd.DataFrame(rows))

info_configs = [
    "configs/experiments/hawkes_jump_info_cvae_logreturn_identity.yaml",
    "configs/experiments/hawkes_jump_info_cvae_logreturn_identity_seed1.yaml",
    "configs/experiments/hawkes_jump_info_cvae_logreturn_identity_seed2.yaml",
]
display(pd.DataFrame({"info_cvae_ablation_config": info_configs, "exists": [repo_path(path).exists() for path in info_configs]}))


## Smoke Commands

Commands are printed by default. Set `RUN_TRAINING=True` or `RUN_EVALUATION=True` locally only when you intentionally want to create ignored artefacts under `outputs/`.


In [ ]:
dry_run_training_command = [
    "poetry",
    "run",
    "tcvae-train",
    "--config",
    display_path(CONFIG_PATH),
    "--output-dir",
    "outputs/hawkes_jump_continuous_logreturn_identity/seed0",
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
smoke_training_command = [
    "poetry",
    "run",
    "tcvae-train",
    "--config",
    display_path(CONFIG_PATH),
    "--output-dir",
    "outputs/hawkes_jump_continuous_logreturn_identity/seed0",
    "--epochs",
    "1",
    "--no-wandb",
]
evaluator_command = [
    "poetry",
    "run",
    "python",
    "scripts/evaluate_hawkes_jump_continuous.py",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    str(CONTINUOUS_MODEL_DIR),
    "--output-dir",
    str(EVALUATION_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--seed",
    "0",
]
fallback_evaluator_command = [
    "poetry",
    "run",
    "tcvae-evaluate",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    str(CONTINUOUS_MODEL_DIR),
    "--output-dir",
    str(EVALUATION_DIR),
    "--n-sample-test",
    str(N_SAMPLE),
    "--seed",
    "0",
]

command_rows = [
    {"purpose": "dry-run training", "command": shell_join(dry_run_training_command)},
    {"purpose": "one-epoch smoke training", "command": shell_join(smoke_training_command)},
    {"purpose": "continuous Hawkes evaluator", "command": shell_join(evaluator_command)},
]
if not repo_path("scripts/evaluate_hawkes_jump_continuous.py").exists():
    command_rows.append(
        {
            "purpose": "available generic evaluator fallback",
            "command": shell_join(fallback_evaluator_command),
        }
    )

display(pd.DataFrame(command_rows))

if RUN_TRAINING:
    raise RuntimeError("RUN_TRAINING is intentionally guarded. Run the printed commands in a terminal when needed.")
if RUN_EVALUATION:
    raise RuntimeError("RUN_EVALUATION is intentionally guarded. Run the printed evaluator command in a terminal when needed.")


## Optional Local-Output Analysis

When `evaluation_summary.json` exists locally, the notebook displays smooth path metrics, jump metrics, and VaR/ES. Missing files are reported with the command that would create them.


In [ ]:
summary_path = EVALUATION_DIR / "evaluation_summary.json"
summary = read_json_if_present(summary_path)
if summary is None:
    display(
        Markdown(
            f"No local summary found at `{display_path(repo_path(summary_path))}`. "
            f"Generate it with:\n\n```bash\n{shell_join(evaluator_command)}\n```"
        )
    )
else:
    metrics = summary.get("metrics", summary)
    if not isinstance(metrics, Mapping):
        metrics = {}
    smooth_rows = [
        {"metric": "MMD", "value": format_metric(metrics, "mmd")},
        {"metric": "SWD", "value": format_metric(metrics, "swd")},
        {"metric": "Terminal W1", "value": format_metric(metrics, "terminal_wasserstein")},
        {"metric": "Volatility W1", "value": format_metric(metrics, "volatility_wasserstein")},
        {"metric": "Drawdown W1", "value": format_metric(metrics, "drawdown_wasserstein")},
    ]
    jump_rows = [
        {"metric": "Jump-count W1", "value": format_metric(metrics, "jump_count_wasserstein")},
        {"metric": "Inter-arrival W1", "value": format_metric(metrics, "inter_arrival_wasserstein")},
        {"metric": "Jump-size W1", "value": format_metric(metrics, "jump_size_wasserstein")},
        {"metric": "Generated jump count", "value": format_metric(metrics, "generated_jump_count_mean")},
        {"metric": "Negative jump fraction", "value": format_metric(metrics, "negative_jump_fraction")},
    ]
    risk_rows = [
        {"metric": "VaR 1%", "value": format_metric(metrics, "var_01", "lower_tail_var_q01")},
        {"metric": "ES 1%", "value": format_metric(metrics, "es_01", "lower_tail_es_q01")},
    ]
    display(Markdown("### Smooth Metrics"))
    display(pd.DataFrame(smooth_rows))
    display(Markdown("### Jump Metrics"))
    display(pd.DataFrame(jump_rows))
    display(Markdown("### VaR/ES"))
    display(pd.DataFrame(risk_rows))


## Optional Evaluation-Batch Plots

Plots are skipped unless `RUN_HEAVY=True` and a local `evaluation_batch.pt` exists. This keeps notebook execution lightweight on clean checkouts.


In [ ]:
batch_path = EVALUATION_DIR / "evaluation_batch.pt"
batch = load_torch_mapping_if_present(batch_path)
if batch is None:
    display(
        Markdown(
            f"No local evaluation batch loaded at `{display_path(repo_path(batch_path))}`. "
            "Set `RUN_HEAVY=True` after generating local outputs to enable plots."
        )
    )
else:
    real = batch.get("real_prices") or batch.get("real_data")
    generated = batch.get("generated_prices") or batch.get("fake_data") or batch.get("decoded_paths")
    if isinstance(real, torch.Tensor) and isinstance(generated, torch.Tensor):
        real_2d = tensor_2d(real)
        generated_2d = tensor_2d(generated)
        fig, ax = plt.subplots(figsize=(9, 4.2))
        for path in real_2d[:8]:
            ax.plot(path.numpy(), color="black", alpha=0.35, linewidth=0.9)
        for path in generated_2d[:8]:
            ax.plot(path.numpy(), alpha=0.7, linewidth=0.9)
        ax.set_title("Hawkes continuous comparator sample paths")
        ax.set_xlabel("Time step")
        ax.set_ylabel("Path value")
        display(fig)
        plt.close(fig)

        returns_real = real_2d.diff(dim=1).reshape(-1)
        returns_generated = generated_2d.diff(dim=1).reshape(-1)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(returns_real.numpy(), bins=60, alpha=0.45, density=True, label="real", color="black")
        ax.hist(returns_generated.numpy(), bins=60, alpha=0.55, density=True, label="generated")
        ax.set_title("One-step log-return proxy histogram")
        ax.legend()
        display(fig)
        plt.close(fig)

    jumps_real = batch.get("real_jumps")
    jumps_generated = batch.get("generated_jumps")
    if isinstance(jumps_real, torch.Tensor) and isinstance(jumps_generated, torch.Tensor):
        counts_real = tensor_2d(jumps_real).sum(dim=1).numpy()
        counts_generated = tensor_2d(jumps_generated).sum(dim=1).numpy()
        fig, ax = plt.subplots(figsize=(7, 4))
        max_count = int(max(counts_real.max(initial=0), counts_generated.max(initial=0)))
        bins = range(max_count + 2)
        ax.hist(counts_real, bins=bins, alpha=0.45, label="real", color="black")
        ax.hist(counts_generated, bins=bins, alpha=0.55, label="generated")
        ax.set_title("Detected jump-count histogram")
        ax.legend()
        display(fig)
        plt.close(fig)

    metrics = batch.get("metrics", {})
    if isinstance(metrics, Mapping):
        display(pd.DataFrame([
            {"metric": "VaR 1%", "value": format_metric(metrics, "var_01", "lower_tail_var_q01")},
            {"metric": "ES 1%", "value": format_metric(metrics, "es_01", "lower_tail_es_q01")},
        ]))


## Optional Torchview

Torchview is intentionally off by default. Enable `RUN_HEAVY=True` locally and instantiate a model from the config before drawing if an architecture diagram is needed.


In [ ]:
if draw_graph is None:
    display(Markdown("`torchview` is not installed; diagram generation is skipped."))
elif not RUN_HEAVY:
    display(Markdown("Torchview diagram skipped because `RUN_HEAVY=False`."))
else:
    display(Markdown("Instantiate the configured continuous model locally before calling `draw_graph`."))
